<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 5: *Subset and Split*
##### Version Number: 4.0
---
### Contents
> *One Hot Encoding*\
> *Separate Dates for modeling*\
> *Downsize Data*\
> *Split Data*\
> *Export Data*
---
### Notes
---
### Inputs
- engineered_samples.csv - Complete main dataset
---
### Outputs 
- `X_ignition`,`X_damage`, `X_spread`, `pal_X` - numeric and bool columns for modeling
- `y_ignition`,`y_spread`,`y_damage` - target data
- `ignition_details`,`spread_details`,`damage_details`,`pal_details` - text columns and spatial info
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [ ]:
# Core data tools
import pandas as pd
import numpy as np
from datetime import datetime

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

---

###  Load Data

In [3]:
samples = pd.read_csv("../data/processed/engineered_samples.csv")

## Original Full Set Distributions

In [ ]:
def make_table(title, series):
    table = Table(title=title)
    table.add_column("Index")
    table.add_column("Value", justify="right")

    for idx, val in series.items():
        t.add_row(str(idx), str(val))

    return table


In [ ]:
display_values(samples['Target_Ignition'].value_counts(),
    samples['Target_Spread'].value_counts(),
    samples['Target_Damage'].value_counts())

## One Hot Encoding

One hot encode appropriate categorical variables.

In [7]:
categorical_columns = ['dominant_province_description','dominant_section_description','Season']
one_hot = samples[categorical_columns]
samples = samples.drop(columns = ['dominant_province_description','dominant_section_description','Season'])

one_hot_samples = pd.get_dummies(
    one_hot,
    columns=categorical_columns,
    drop_first=False
).astype(int)

encoded_samples = pd.concat([samples,one_hot_samples],axis=1)

### Separate Dates for modeling and case study

- **01/01/2018 - 12/31/2024** Dates to train the models
- **01/01/2025 - 01/23/2025** Dates for evaluating model performance on unseen data

In [8]:
encoded_samples['Date'] = pd.to_datetime(encoded_samples['Date'])

# first day to analyze in weather dataset
FIRST_DATE = pd.to_datetime('2018-01-01').date()

# last day to analyze in weather dataset
LAST_DATE = pd.to_datetime('2024-12-31').date()

# Boolean mask for dates within the range
mask = (encoded_samples['Date'].dt.date >= FIRST_DATE) & (encoded_samples['Date'].dt.date <= LAST_DATE)

pal = encoded_samples.loc[~mask]

# Keep only rows within the range
model_samples = encoded_samples.loc[mask].copy()

### (OPTIONAL) Subset classes to save processor
- Downsize the majority class in all three sets

In [ ]:
model_samples['Date'] = pd.to_datetime(model_samples['Date'])

ignition_reduced = subset_df_cap_majority(model_samples,'Target_Ignition',30000)
spread_reduced = subset_df_cap_majority(model_samples,'Target_Spread',30000)
damage_reduced = subset_df_cap_majority(model_samples,'Target_Damage',30000)

display_values(ignition_reduced['Target_Ignition'].value_counts(),
    spread_reduced['Target_Spread'].value_counts(),
    damage_reduced['Target_Damage'].value_counts())

## Split Data

In [ ]:
text_columns = ['Sample_ID', 'Date', 'grid_id',
       'geometry','area_in_cali','maximum_x', 'minimum_y',
       'maximum_y', 'minimum_x','centroid_northing','centroid_easting','Target_Damage',
                'Target_Ignition', 'Target_Spread','Year']

## Modeling Data
y_ignition = ignition_reduced['Target_Ignition']
y_spread = spread_reduced['Target_Spread']
y_damage = damage_reduced['Target_Damage']

X_ignition = ignition_reduced.drop(columns=text_columns)
X_spread = spread_reduced.drop(columns=text_columns)
X_damage = damage_reduced.drop(columns=text_columns)

details_ignition = ignition_reduced[text_columns]
details_spread = spread_reduced[text_columns]
details_damage = damage_reduced[text_columns]

## Case study data
pal_X = pal.drop(columns=text_columns)
pal_y = pal[['Target_Ignition','Target_Spread','Target_Damage']]
pal_details = pal[text_columns]


## Export Data

In [14]:
X_ignition.to_csv('../data/processed/X_ignition.csv', index=False)
X_spread.to_csv('../data/processed/X_spread.csv', index=False)
X_damage.to_csv('../data/processed/X_damage.csv', index=False)

y_ignition.to_csv('../data/processed/y_ignition.csv', index=False)
y_spread.to_csv('../data/processed/y_spread.csv', index=False)
y_damage.to_csv('../data/processed/y_damage.csv', index=False)

details_ignition.to_csv('../data/processed/details_ignition.csv', index=False)
details_spread.to_csv('../data/processed/details_spread.csv', index=False)
details_damage.to_csv('../data/processed/details_damage.csv', index=False)

pal_X.to_csv('../data/processed/pal_X.csv', index=False)
pal_y.to_csv('../data/processed/pal_y.csv', index=False)
pal_details.to_csv('../data/processed/pal_details.csv', index=False)

print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
